# 神經網路經典架構

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab7.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab7.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

## Torch Hub 預訓練模型推論與 TensorBoard 視覺化

本章示範如何：
1. 匯入必要套件。
2. 下載範例圖片並完成與 ImageNet 相容的前處理。
3. 使用 `torch.hub.load` 載入多個預訓練模型。
4. 進行推論、輸出 Top-5 類別機率，並將模型圖寫入 TensorBoard。

import 基本套件

In [ ]:
import torch
import torch.hub
import torchvision
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary
from PIL import Image

下載範例圖片並建立圖片預處理



In [ ]:
# 定義 device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 從 pytorch 官方 Github 下載 "狗" 類別的圖片, 存到 'dog.jpg'
import urllib.request
url, filename = "https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg"
try:
    urllib.request.urlretrieve(url, filename)
except Exception as e:
    print(f"Error: Unable to retrieve {url} to {filename} ({e})")

# 開啟圖片檔
img_input = Image.open(filename)

# 預處理函式的定義: 改變圖片大小、截取圖片中間區域、張量化、標準化
# 注意這邊的預處理需要依照預訓練模型的預處理方式進行
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(299),
    torchvision.transforms.CenterCrop(299),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 對圖片進行預處理
image = transform(img_input).unsqueeze(0)
image = image.to(device)
print(f"Input tensor shape: {tuple(image.shape)}")

載入模型，列印網路架構，進行推論


In [ ]:
# 3) 使用 torch.hub.load 來載入不同的網路模型
model_names = ["inception_v3", "resnet50", "mobilenet_v3_small"]

for model_name in model_names:
    print("=" * 80)
    print(f"載入模型: {model_name}")

    # 注意: 不同 torchvision 版本的參數可能不同，先嘗試 weights，再 fallback 到 pretrained
    try:
        model = torch.hub.load(
            "pytorch/vision:v0.10.0",
            model_name,
            weights="IMAGENET1K_V1"
        )
    except TypeError:
        model = torch.hub.load(
            "pytorch/vision:v0.10.0",
            model_name,
            pretrained=True
        )

    model = model.to(device)
    model.eval()

    print(f"{model_name} 網路架構：")
    input_size = (1, 3, 299, 299)
    summary(model, input_size=input_size, device=str(device))

    with torch.no_grad():
        output = model(image)

    writer = SummaryWriter(f"runs/{model_name}")
    writer.add_graph(model, image)
    writer.close()

    probs = torch.nn.functional.softmax(output, dim=1)
    top_k = torch.topk(probs, k=5)
    classes = top_k.indices[0].tolist()
    probabilities = top_k.values[0].tolist()

    print(f"Top 5 classes are: {classes}")
    print(f"Estimated probs: {probabilities}")

### 查看 TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs